# Tutorial: MNIST Checkpoint-Replay Prototype

This notebook is the MNIST study in the form we would actually want to run it for a paper prototype.

We still keep the setup visible and line by line, but the main path now writes a real attempt directory under an experiment root, saves checkpoints, replays those checkpoints, and plots memorization curves over training time `tau`.


## Outline

1. Choose the study knobs.
2. Load MNIST and build the class-balanced train/reference subsets explicitly.
3. Inspect the images and class counts before training.
4. Build the config in place.
5. Train a real saved run, replay every checkpoint, and then read the written artifacts back from disk.
6. Plot `f_mem` and nearest-neighbor ratio statistics as functions of checkpoint step `tau`.
        

## 0. Knobs

`FAST_NOTEBOOK` can be controlled from the environment with `MEMPAR_FAST_NOTEBOOK=1` when we want a short smoke pass. The notebook defaults to the full prototype shape otherwise.
        

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

FAST_NOTEBOOK = os.environ.get("MEMPAR_FAST_NOTEBOOK", "0") == "1"
DOWNLOAD = os.environ.get("MEMPAR_DOWNLOAD", "1") == "1"

SEED = 7
FRAMEWORK_FAMILY = "fm"
PREDICTION_SPACE = "v"
LOSS_SPACE = "v"
RATIO_THRESHOLD = 0.5

N_TRAIN = 100 if not FAST_NOTEBOOK else 20
N_EVAL = N_TRAIN
N_GENERATED = min(100, N_EVAL) if not FAST_NOTEBOOK else 10

TRAIN_STEPS = 5_000 if not FAST_NOTEBOOK else 200
CKPT_EVERY = 500 if not FAST_NOTEBOOK else 50
SAMPLE_STEPS = 24 if not FAST_NOTEBOOK else 8

BASE_CHANNELS = 128
BATCH_SIZE = min(16, N_TRAIN)
RUN_NAME = (
    "mnist_checkpoint_replay_fast" if FAST_NOTEBOOK else "mnist_checkpoint_replay"
)
EXPERIMENT_DIR = Path("tmp/mnist_checkpoint_replay")
DATA_ROOT = Path("tmp/data")
ATTEMPT_DIR = None  # Set this to an existing attempt dir to skip training and inspect/replay it.

LOCAL_BASELINE_SECONDS = 7.11
LOCAL_BASELINE_STEPS = 50


In [ ]:
TRAIN_STEPS

## 1. Imports and repo path setup

The notebook lives under `experiments/mnist/`, so we add both the repo root and `src/` to `sys.path` explicitly before importing the canonical training and metric stack.
        

In [ ]:
import json
import random
import sys
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchvision import datasets

repo_root = Path.cwd().resolve()
for candidate in (repo_root, *repo_root.parents):
    if (candidate / "src").exists() and (candidate / "experiments").exists():
        repo_root = candidate
        break
else:
    raise RuntimeError(
        "Could not find the repo root from the current working directory."
    )

src_root = repo_root / "src"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from mp_artifacts import load_json, update_attempt_status
from mp_config import get_base_config
from mp_metrics import evaluate_run
from mp_training import train_run


def show_image_grid(images, labels=None, title="", ncol=10, value_range=None):
    images = images.detach().cpu()
    if images.dim() == 4:
        images = images.squeeze(1)
    n_show = int(images.shape[0])
    ncol = min(ncol, n_show)
    nrow = (n_show + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(1.15 * ncol, 1.3 * nrow))
    if hasattr(axes, "reshape"):
        axes = list(axes.reshape(-1))
    else:
        axes = [axes]

    for idx, ax in enumerate(axes):
        ax.axis("off")
        if idx >= n_show:
            continue
        kwargs = {}
        if value_range is not None:
            kwargs["vmin"], kwargs["vmax"] = value_range
        ax.imshow(images[idx], cmap="gray", **kwargs)
        if labels is not None:
            ax.set_title(f"y={int(labels[idx])}", fontsize=8)

    if title:
        fig.suptitle(title, fontsize=12)
    fig.tight_layout()
    return fig


def format_duration(seconds):
    seconds = float(seconds)
    hours = seconds / 3600.0
    minutes = seconds / 60.0
    return f"{seconds:,.1f}s  ({minutes:,.1f} min, {hours:,.2f} h)"

## 2. Make the run deterministic and choose the device

The full prototype is intended for local CPU or MPS first. Step cost should depend mainly on model width, batch size, image size, and device, not on `N_train` once the subset is already materialized in memory.
        

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

requested_device = os.environ.get("MEMPAR_DEVICE")
if requested_device:
    DEVICE = requested_device
elif torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

data_root = (repo_root / DATA_ROOT).resolve()
experiment_dir = (repo_root / EXPERIMENT_DIR).resolve()
resolved_attempt_dir = (Path(ATTEMPT_DIR).expanduser().resolve() if ATTEMPT_DIR else None)

action_summary = {
    "FAST_NOTEBOOK": FAST_NOTEBOOK,
    "DEVICE": DEVICE,
    "N_TRAIN": N_TRAIN,
    "N_EVAL": N_EVAL,
    "N_GENERATED": N_GENERATED,
    "TRAIN_STEPS": TRAIN_STEPS,
    "CKPT_EVERY": CKPT_EVERY,
    "SAMPLE_STEPS": SAMPLE_STEPS,
    "RUN_NAME": RUN_NAME,
    "experiment_dir": str(experiment_dir),
    "attempt_dir": str(resolved_attempt_dir) if resolved_attempt_dir else None,
    "data_root": str(data_root),
}
for key, value in action_summary.items():
    print(f"{key:>14}: {value}")


## 3. Load the raw MNIST splits

We open the raw torchvision splits first so the subset construction is transparent.
        

In [ ]:
train_set = datasets.MNIST(root=str(data_root), train=True, download=DOWNLOAD)
test_set = datasets.MNIST(root=str(data_root), train=False, download=DOWNLOAD)

print(f"torchvision train split size : {len(train_set.data):,}")
print(f"torchvision test split size  : {len(test_set.data):,}")
print(f"raw train image dtype        : {train_set.data.dtype}")
print(f"raw label dtype              : {train_set.targets.dtype}")

## 4. Build the balanced train and reference subsets explicitly

The saved-run loader will rebuild these same subsets from the config, but we write them out here first so we can inspect them before the training job starts.
        

In [ ]:
num_classes = 10
if N_TRAIN % num_classes != 0:
    raise ValueError(f"N_TRAIN must be divisible by {num_classes}, got {N_TRAIN}")
if N_EVAL % num_classes != 0:
    raise ValueError(f"N_EVAL must be divisible by {num_classes}, got {N_EVAL}")

train_per_class = N_TRAIN // num_classes
reference_per_class = N_EVAL // num_classes

train_generator = torch.Generator().manual_seed(SEED)
reference_generator = torch.Generator().manual_seed(SEED + 1)

train_index_chunks = []
for class_id in range(num_classes):
    class_indices = torch.nonzero(
        train_set.targets == class_id, as_tuple=False
    ).flatten()
    if class_indices.numel() < train_per_class:
        raise ValueError(f"Not enough training images for class {class_id}")
    chosen = torch.randperm(class_indices.numel(), generator=train_generator)[
        :train_per_class
    ]
    train_index_chunks.append(class_indices[chosen])
train_indices = torch.cat(train_index_chunks)
train_indices = train_indices[
    torch.randperm(train_indices.numel(), generator=train_generator)
]

reference_index_chunks = []
for class_id in range(num_classes):
    class_indices = torch.nonzero(
        test_set.targets == class_id, as_tuple=False
    ).flatten()
    if class_indices.numel() < reference_per_class:
        raise ValueError(f"Not enough reference images for class {class_id}")
    chosen = torch.randperm(class_indices.numel(), generator=reference_generator)[
        :reference_per_class
    ]
    reference_index_chunks.append(class_indices[chosen])
reference_indices = torch.cat(reference_index_chunks)
reference_indices = reference_indices[
    torch.randperm(reference_indices.numel(), generator=reference_generator)
]

raw_train_images = train_set.data[train_indices].clone()
raw_reference_images = test_set.data[reference_indices].clone()
train_labels = train_set.targets[train_indices].clone()
reference_labels = test_set.targets[reference_indices].clone()

## 5. Inspect class counts and example images

Before training, we check that both saved subsets are exactly balanced and that the selected examples look reasonable.
        

In [ ]:
train_class_counts = torch.bincount(train_labels, minlength=num_classes)
reference_class_counts = torch.bincount(reference_labels, minlength=num_classes)

print("train class counts     :", train_class_counts.tolist())
print("reference class counts :", reference_class_counts.tolist())
print("first train indices    :", train_indices[:10].tolist())
print("first reference indices:", reference_indices[:10].tolist())

assert torch.equal(train_class_counts, torch.full((num_classes,), train_per_class))
assert torch.equal(
    reference_class_counts, torch.full((num_classes,), reference_per_class)
)

In [ ]:
preview_count = min(30, raw_train_images.shape[0])
show_image_grid(
    raw_train_images[:preview_count],
    labels=train_labels[:preview_count],
    title="Balanced MNIST training subset (raw uint8 images)",
    ncol=10,
    value_range=(0, 255),
)
plt.show()

## 6. Normalize to `[-1, 1]` and resize to 32x32 for the UNet

The real run will do this inside the canonical loader. We do it once here too so we can inspect the actual tensor geometry the UNet will see.
        

In [ ]:
train_preview_tensor = raw_train_images.float().div(127.5).sub(1.0).unsqueeze(1)
reference_preview_tensor = raw_reference_images.float().div(127.5).sub(1.0).unsqueeze(1)

train_preview_tensor = F.interpolate(
    train_preview_tensor,
    size=(32, 32),
    mode="bilinear",
    align_corners=False,
)
reference_preview_tensor = F.interpolate(
    reference_preview_tensor,
    size=(32, 32),
    mode="bilinear",
    align_corners=False,
)

print("train tensor shape     :", tuple(train_preview_tensor.shape))
print("reference tensor shape :", tuple(reference_preview_tensor.shape))
print(
    "train tensor range     :",
    (float(train_preview_tensor.min()), float(train_preview_tensor.max())),
)

## 7. Build the saved-run config line by line

This is the same configuration shape the final script will use. The key difference from the earlier notebook is that we will now hand this config to the canonical `train_run(...)` and `evaluate_run(...)` functions instead of running a purely in-memory loop.
        

In [ ]:
config = get_base_config()

config.study.name = "mnist"
config.study.description = (
    "MNIST checkpoint-replay prototype written from the notebook."
)

config.framework.family = FRAMEWORK_FAMILY
config.framework.prediction_space = PREDICTION_SPACE
config.framework.loss_space = LOSS_SPACE

config.data.dataset = "mnist"
config.data.root = str(data_root)
config.data.download = DOWNLOAD
config.data.n_train = int(N_TRAIN)
config.data.n_eval = int(N_EVAL)
config.data.image_size = 32
config.data.image_channels = 1
config.data.flatten = False
config.data.data_dim = 32 * 32
config.data.balance_train_classes = True
config.data.balance_reference_classes = True

config.network.arch = "unet"
config.network.base_channels = BASE_CHANNELS
config.network.channel_mult = (1, 2, 4)
config.network.attn_resolutions = (16, 8)
config.network.num_res_blocks = 2
config.network.dropout = 0.1
config.network.time_emb_dim = None

config.training.seed = int(SEED)
config.training.optimizer = "sgd"
config.training.lr = 1.0e-2
config.training.momentum = 0.95
config.training.n_steps = int(TRAIN_STEPS)
config.training.batch_size = int(BATCH_SIZE)
config.training.ckpt_every = int(CKPT_EVERY)
config.training.log_every = max(1, min(500, CKPT_EVERY // 2))

config.evaluation.n_samples = int(N_GENERATED)
config.evaluation.batch_size = int(min(BATCH_SIZE, N_GENERATED))
config.evaluation.ratio_threshold = float(RATIO_THRESHOLD)
config.evaluation.sample_panel_count = min(8, int(N_GENERATED))
config.evaluation.chunk_size = int(max(128, N_TRAIN, N_EVAL, N_GENERATED))

config.sampling.method = "heun"
config.sampling.n_steps = int(SAMPLE_STEPS)

config.outputs.run_name = RUN_NAME

config

## 8. Estimate runtime before launching the run

This is the earlier local baseline estimate. It is only a prototype estimate, not a final cluster-time claim.
        

In [ ]:
local_seconds_per_step = LOCAL_BASELINE_SECONDS / LOCAL_BASELINE_STEPS
baseline_30k = 30_000 * local_seconds_per_step
baseline_1e6 = 1_000_000 * local_seconds_per_step

print(f"baseline seconds/step : {local_seconds_per_step:.4f}")
print(f"baseline 30k steps    : {format_duration(baseline_30k)}")
print(f"baseline 1e6 steps    : {format_duration(baseline_1e6)}")
print(
    "note                  : once the subset is materialized, N_train mostly affects upfront data work and saved artifacts, not the steady-state per-step compute."
)

## 9. Train the saved attempt and write checkpoints

This is now the canonical training path. It writes an attempt directory under `tmp/mnist_checkpoint_replay/runs/<run_id>/seeds/<seed_id>/attempts/<attempt_id>/` and stores the replay-critical tensors, labels, indices, checkpoints, manifests, and status metadata on disk.


In [ ]:
if resolved_attempt_dir is None:
    experiment_dir.mkdir(parents=True, exist_ok=True)
    run_dir = train_run(config, experiment_dir, device=DEVICE)
else:
    run_dir = resolved_attempt_dir

print("run_dir:", run_dir)
print("checkpoints:", sorted((run_dir / "checkpoints").glob("step_*.pt"))[:3], "...")


## 10. Replay every checkpoint into memorization metrics

We now loop over the saved checkpoints with the official replay code and score the generated samples against the saved training tensor. The same pass also scores the held-out reference subset against that same training tensor as a baseline.
        

In [ ]:
records = evaluate_run(run_dir, device=DEVICE)
print(f"replayed checkpoints : {len(records)}")
print("last record          :")
print(json.dumps(records[-1], indent=2))

## 11. Read the written artifacts back from disk

This is the key change from the earlier notebook: the run itself is now the source of truth. We reload the saved metadata, labels, and indices and verify that they match the balanced subset we constructed at the top.
        

In [ ]:
artifacts_dir = run_dir / "artifacts"
metrics_dir = run_dir / "metrics"
figures_dir = run_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
status = load_json(run_dir / "status.json")

data_meta = load_json(artifacts_dir / "data_meta.json")
run_summary = load_json(artifacts_dir / "run_summary.json")
checkpoint_metrics = load_json(metrics_dir / "checkpoint_metrics.json")

saved_train_indices = torch.load(artifacts_dir / "train_indices.pt", map_location="cpu")
saved_reference_indices = torch.load(
    artifacts_dir / "reference_indices.pt", map_location="cpu"
)
saved_train_labels = torch.load(artifacts_dir / "train_labels.pt", map_location="cpu")
saved_reference_labels = torch.load(
    artifacts_dir / "reference_labels.pt", map_location="cpu"
)

assert torch.equal(saved_train_indices, train_indices)
assert torch.equal(saved_reference_indices, reference_indices)
assert torch.equal(saved_train_labels, train_labels)
assert torch.equal(saved_reference_labels, reference_labels)

print(json.dumps(status, indent=2))
print(json.dumps(data_meta, indent=2))
print(json.dumps(run_summary, indent=2))


## 12. Update the runtime estimate with the actual observed run

For a real long run, this observed `training_seconds_per_step` is the number we would extrapolate from first.
        

In [ ]:
observed_seconds_per_step = float(run_summary["training_seconds_per_step"])
observed_30k = 30_000 * observed_seconds_per_step
observed_1e6 = 1_000_000 * observed_seconds_per_step

print(
    f"observed training wall time : {format_duration(run_summary['training_wall_time_sec'])}"
)
print(f"observed seconds/step       : {observed_seconds_per_step:.6f}")
print(f"observed 30k estimate       : {format_duration(observed_30k)}")
print(f"observed 1e6 estimate       : {format_duration(observed_1e6)}")

## 13. Plot memorization curves over training time `tau`

We use the saved `checkpoint_metrics.json` file here, not the in-memory `records` list, so the plotting cell reflects the actual on-disk contract.
        

In [ ]:
steps = [record["step"] for record in checkpoint_metrics]
f_mem = [record["f_mem"] for record in checkpoint_metrics]
baseline_f_mem = [record["baseline_f_mem"] for record in checkpoint_metrics]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(steps, f_mem, marker="o", label="generated samples")
ax.plot(steps, baseline_f_mem, marker="s", label="held-out reference baseline")
ax.set_xlabel("tau (training step)")
ax.set_ylabel("f_mem")
ax.set_title("Memorization fraction across checkpoints")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

f_mem_path = figures_dir / f"{run_dir.name}_f_mem_vs_tau.png"
fig.savefig(f_mem_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", f_mem_path)

In [ ]:
mean_ratio = [record["mean_nn_ratio"] for record in checkpoint_metrics]
std_ratio = [record["std_nn_ratio"] for record in checkpoint_metrics]
baseline_mean_ratio = [
    record["baseline_mean_nn_ratio"] for record in checkpoint_metrics
]
baseline_std_ratio = [record["baseline_std_nn_ratio"] for record in checkpoint_metrics]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(
    steps, mean_ratio, color="tab:blue", marker="o", label="generated mean NN ratio"
)
ax.fill_between(
    steps,
    [m - s for m, s in zip(mean_ratio, std_ratio)],
    [m + s for m, s in zip(mean_ratio, std_ratio)],
    color="tab:blue",
    alpha=0.2,
)
ax.plot(
    steps,
    baseline_mean_ratio,
    color="tab:orange",
    marker="s",
    label="held-out reference mean NN ratio",
)
ax.fill_between(
    steps,
    [m - s for m, s in zip(baseline_mean_ratio, baseline_std_ratio)],
    [m + s for m, s in zip(baseline_mean_ratio, baseline_std_ratio)],
    color="tab:orange",
    alpha=0.2,
)
ax.set_xlabel("tau (training step)")
ax.set_ylabel("NN ratio")
ax.set_title("Nearest-neighbor ratio statistics across checkpoints")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

ratio_path = figures_dir / f"{run_dir.name}_nn_ratio_vs_tau.png"
fig.savefig(ratio_path, dpi=180, bbox_inches="tight")
plt.show()
print("saved:", ratio_path)

update_attempt_status(
    run_dir,
    "completed",
    generated_figures={
        "f_mem": str(f_mem_path),
        "nn_ratio": str(ratio_path),
    },
)


## 14. Inspect the final sample-vs-nearest-neighbor panel

The replay code saves a small panel payload at the final checkpoint so we can look at actual examples instead of only scalar curves.
        

In [ ]:
final_samples = torch.load(metrics_dir / "final_samples.pt", map_location="cpu")
final_neighbors = torch.load(metrics_dir / "final_neighbors.pt", map_location="cpu")

show_image_grid(
    final_samples,
    title="Final checkpoint: generated samples",
    ncol=min(8, final_samples.shape[0]),
    value_range=(-1, 1),
)
plt.show()

show_image_grid(
    final_neighbors,
    title="Nearest training neighbors for the final generated panel",
    ncol=min(8, final_neighbors.shape[0]),
    value_range=(-1, 1),
)
plt.show()

## 15. What this prototype leaves for later

- We are still doing one run at one `N_train` here.
- We are not doing sweep aggregation or W&B logging yet.
- We are showing within-checkpoint standard deviations for NN ratios, not cross-seed uncertainty.

That is enough for the current milestone: one balanced MNIST run, written as a real artifact-producing workflow, with checkpoint replay and memorization curves over `tau`.
        

In [ ]:
summary_lines = [
    f"run_dir            : {run_dir}",
    f"status             : {run_dir / 'status.json'}",
    f"checkpoint_metrics : {metrics_dir / 'checkpoint_metrics.json'}",
    f"train_history      : {metrics_dir / 'train_history.json'}",
    f"data_meta          : {artifacts_dir / 'data_meta.json'}",
    f"run_summary        : {artifacts_dir / 'run_summary.json'}",
    f"f_mem figure       : {f_mem_path}",
    f"ratio figure       : {ratio_path}",
]
print("\n".join(summary_lines))
